In [ ]:
import pandas as pd
from pathlib import Path

_current_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    path for path in (_current_dir, *_current_dir.parents)
    if (path / "notebooks").is_dir() and (path / "data").is_dir()
)
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"


## SOL 지갑별 송금·수신량 집계

여러 전송 CSV를 결합한 뒤 pandas 그룹 연산으로 지갑별 송금액과 수신액을 계산합니다.


In [ ]:
def summarize_wallet_flows(paths):
    transactions = pd.concat((pd.read_csv(path) for path in paths), ignore_index=True)
    sent = transactions.groupby("sender_address")["amount"].sum().rename("sent")
    received = transactions.groupby("receiver_address")["amount"].sum().rename("received")
    return pd.concat([sent, received], axis=1).fillna(0).rename_axis("address").reset_index()


In [ ]:
input_dir = DATA_DIR / "processed" / "solana"
input_paths = sorted(input_dir.glob("*TRX.csv"))
output_dir = RESULTS_DIR / "solana" / "wallet_analysis"
output_dir.mkdir(parents=True, exist_ok=True)

if input_paths:
    wallet_flows = summarize_wallet_flows(input_paths)
    wallet_flows.to_csv(output_dir / "sol_wallet_flows.csv", index=False)
else:
    print(f"No transaction files found in {input_dir}")
